# Compare CLIP segment aggregation methods

This notebook tests how to combine CLIP frame predictions for **one detected dog vocalization**. It compares average frame probabilities, average logits, confidence-weighted logits, and majority voting on the same detected events. Bark detection itself uses the project configuration; the compared settings change **scene aggregation**, not the bark detector.

The supplied annotations label a *time span*, not each bark. A contained event inherits that span's category as a **weak proxy**, which can be wrong if the situation changes or the detector finds a false alarm. Report results as exploratory, not verified bark-level accuracy. These videos are a **testing set**: choose configurations in advance and do not tune model settings or fit a temperature on their results. Scores are uncalibrated and not a translation of dog intent.

Run from the project root or `notebooks` folder in the project's Python 3.12 kernel. Install the project with `pip install -e ".[ml,dev]"`. Bark detection requires the existing PretrainedSED source and its checkpoint; see `src/woofdecoder/preprocess/bark_detector/pretrained_sed.py` and notebooks 12-14. CLIP and detector weights must be cached or downloadable. No input video, labels, or model config is modified; results remain in memory.


In [ ]:
import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/woofdecoder/taxonomy.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start the notebook inside the project or its notebooks folder.")
sys.path.insert(0, str(ROOT / "src"))
# Optional local dependency directory, used only when torchaudio is not installed.
local_deps = ROOT / "third_party/scene_notebook_deps"
if importlib.util.find_spec("torchaudio") is None and local_deps.is_dir():
    sys.path.insert(0, str(local_deps))

from woofdecoder.config import load_config
from woofdecoder.preprocess.bark_detector import (
    detect_sed_probs,
    find_dog_events,
    load_bark_detector,
    settings_from_config,
    vocal_score,
)
from woofdecoder.preprocess.media import load_audio_16k, sample_frames
from woofdecoder.preprocess.segments import BarkSegment
from woofdecoder.scene import SceneBranch
from woofdecoder.taxonomy import CATEGORIES

print("Project:", ROOT)
print("Categories:", CATEGORIES)

## 1. Choose inputs and experiment settings

By default all label rows and their videos are included. Set `MAX_VIDEOS` to a positive number for a quick smoke run; `None` means all. Keep the testing data separate from branch-training dogs and select the methods and temperatures **before** inspecting test results. Source-video timestamps in the CSV are used as written. Video names are matched case-insensitively by `video_id` and `.mp4` suffix.


In [ ]:
LABELS_PATH = Path(r"C:/Users/has/Desktop/testing/labels.csv")
VIDEOS_DIR = LABELS_PATH.parent / "videos"
MAX_VIDEOS = None  # e.g. 2 to run only the first two videos
TEMPERATURES = [1.0, 2.0]  # fixed comparison; no fitting on test labels
CONFIDENCE_WEIGHT = 0.5  # alpha, in [0, 1]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if not LABELS_PATH.is_file() or not VIDEOS_DIR.is_dir():
    raise FileNotFoundError(f"Check LABELS_PATH and VIDEOS_DIR: {LABELS_PATH}, {VIDEOS_DIR}")
if MAX_VIDEOS is not None and (
    isinstance(MAX_VIDEOS, bool) or not isinstance(MAX_VIDEOS, int) or MAX_VIDEOS < 1
):
    raise ValueError("MAX_VIDEOS must be None or a positive integer")
if not TEMPERATURES or any(not np.isfinite(t) or t <= 0 for t in TEMPERATURES):
    raise ValueError("Each temperature must be finite and positive")
if len(set(TEMPERATURES)) != len(TEMPERATURES):
    raise ValueError("Temperatures must be unique")
if not np.isfinite(CONFIDENCE_WEIGHT) or not 0 <= CONFIDENCE_WEIGHT <= 1:
    raise ValueError("CONFIDENCE_WEIGHT must be between 0 and 1")

labels = pd.read_csv(LABELS_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig")
required = {"video_id", "dog_id", "start_s", "end_s", "category"}
if missing := sorted(required - set(labels.columns)):
    raise ValueError(f"Missing labels.csv columns: {missing}")
labels.insert(0, "label_row", np.arange(len(labels)))  # original zero-based data row
for col in ["video_id", "dog_id", "category"]:
    labels[col] = labels[col].str.strip()
for col in ["start_s", "end_s"]:
    labels[col] = pd.to_numeric(labels[col], errors="coerce")

valid = (
    labels["video_id"].ne("")
    & labels["dog_id"].ne("")
    & labels["category"].isin(CATEGORIES)
    & np.isfinite(labels["start_s"])
    & np.isfinite(labels["end_s"])
    & labels["start_s"].ge(0)
    & labels["end_s"].gt(labels["start_s"])
)
invalid_labels = labels.loc[~valid].copy()
if not invalid_labels.empty:
    print("Invalid label rows (excluded):")
    display(invalid_labels)
labels = labels.loc[valid].copy()
labels["video_key"] = labels["video_id"].str.casefold()
video_keys = labels["video_key"].drop_duplicates().tolist()
if MAX_VIDEOS is not None:
    video_keys = video_keys[:MAX_VIDEOS]
labels = labels[labels["video_key"].isin(video_keys)].copy()
if labels.empty:
    raise ValueError("No valid labeled rows selected")
video_index = {}
for path in sorted(VIDEOS_DIR.iterdir()):
    if path.is_file() and path.suffix.casefold() == ".mp4":
        video_index.setdefault(path.stem.casefold(), []).append(path)
print(f"Selected {len(labels)} labeled spans from {len(video_keys)} videos; device: {DEVICE}")
print("Annotation counts:")
display(labels.groupby("category").size().reindex(CATEGORIES, fill_value=0).rename("spans"))
print(
    "Duplicate or overlapping spans are retained and treated as ambiguous when an event intersects more than one."
)

## 2. Load the production models

The bark detector runs on the **full source video once**. Frame extraction is limited to detected events that belong to one annotated span. The app's event detector also recognizes growls, whines, howls and other dog vocalizations; it is not a bark-only ground truth detector. `SceneBranch` loads CLIP once, and its raw logits are reused across every aggregation and temperature below.


In [ ]:
config = load_config()
settings = settings_from_config(config)
print("Bark settings:", settings)
try:
    detector = load_bark_detector(config["bark_detector"]["model"], DEVICE, verbose=False)
except Exception as exc:
    raise RuntimeError(
        "Could not load the bark detector. Check PretrainedSED source, checkpoint and ML "
        "dependencies; see notebooks 12-14 and the exception above."
    ) from exc
try:
    scene = SceneBranch.from_config(config, DEVICE)
except Exception as exc:
    raise RuntimeError(
        "Could not load CLIP. Check the configured model and cached weights."
    ) from exc
print("Loaded CLIP model:", scene.encoder.schema["model"])
print("Prompt order:", scene.encoder.schema["categories"])

## 3. Extract and label detected events

An event is scored only if it is **fully contained within exactly one labeled span** and intersects no other labeled span. A boundary-crossing event or an event intersecting multiple annotations is reported but not scored. Events outside all labeled spans, missing files or frames, no-audio/no-event videos, and failures are tracked separately. Detector timestamps use the full source video; frame timestamps are requested sampling positions and short events can repeat video frames.


In [ ]:
def match_event(start, end, spans):
    """Return status and label row; any second overlap makes the event ambiguous."""
    overlaps = spans[(spans["start_s"] < end) & (spans["end_s"] > start)]
    if overlaps.empty:
        return "outside_labeled_spans", None, []
    ids = overlaps["label_row"].tolist()
    if len(overlaps) != 1:
        return "ambiguous_overlapping_labels", None, ids
    row = overlaps.iloc[0]
    if start < row["start_s"] - 1e-6 or end > row["end_s"] + 1e-6:
        return "crosses_label_boundary", None, ids
    return "matched", row, ids


# Report at both event and annotation levels rather than silently dropping anything.
event_records, video_records, scored = [], [], []
for number, key in enumerate(video_keys, 1):
    spans = labels[labels["video_key"] == key]
    paths = video_index.get(key, [])
    print(f"[{number}/{len(video_keys)}] {spans.iloc[0]['video_id']}")
    video_record = {
        "video_id": spans.iloc[0]["video_id"],
        "video_key": key,
        "spans": len(spans),
        "events": 0,
        "status": "ok",
        "detail": "",
    }
    if len(paths) != 1:
        video_record["status"] = "missing_video" if not paths else "ambiguous_video_filename"
        video_record["detail"] = ", ".join(str(p.name) for p in paths)
        video_records.append(video_record)
        continue
    path = paths[0]
    try:
        audio = load_audio_16k(path, verbose=False)
        if audio is None or audio.size == 0:
            video_record["status"] = "no_audio"
            video_records.append(video_record)
            continue
        sed_probs = detect_sed_probs(detector, audio)
        events = find_dog_events(vocal_score(sed_probs), settings)
        video_record["events"] = len(events)
        if not events:
            video_record["status"] = "no_detected_events"
        for start, end in events:
            status, label, overlap_rows = match_event(start, end, spans)
            entry = {
                "video_id": spans.iloc[0]["video_id"],
                "start_s": float(start),
                "end_s": float(end),
                "status": status,
                "overlap_label_rows": overlap_rows,
                "label_row": int(label["label_row"]) if label is not None else None,
                "dog_id": label["dog_id"] if label is not None else None,
                "category": label["category"] if label is not None else None,
                "frame_count": 0,
                "detail": "",
            }
            if status != "matched":
                event_records.append(entry)
                continue
            try:
                frames, times = sample_frames(path, start, end, verbose=False)
                entry["frame_count"] = len(frames)
                if not frames:
                    entry["status"] = "missing_video_frames"
                else:
                    segment = BarkSegment(
                        start_s=start, end_s=end, source_id=key, frames=frames, frame_times_s=times
                    )
                    result = scene.encoder.encode_segment(segment)
                    logits = result["frame_logits"]
                    if (
                        logits.shape != (len(frames), len(CATEGORIES))
                        or not np.isfinite(logits).all()
                    ):
                        raise ValueError("Invalid CLIP frame logits")
                    entry["status"] = "scored"
                    scored.append(
                        {
                            **entry,
                            "frame_logits": logits,
                            "frame_times_s": times,
                            "configured_probs": result["probs"],
                        }
                    )
            except Exception as exc:  # noqa: BLE001
                entry["status"] = "frame_or_clip_error"
                entry["detail"] = f"{type(exc).__name__}: {exc}"
            event_records.append(entry)
    except Exception as exc:  # noqa: BLE001
        video_record["status"] = "audio_or_detector_error"
        video_record["detail"] = f"{type(exc).__name__}: {exc}"
    video_records.append(video_record)

videos_report = pd.DataFrame(video_records)
events_report = pd.DataFrame(
    event_records,
    columns=[
        "video_id",
        "start_s",
        "end_s",
        "status",
        "overlap_label_rows",
        "label_row",
        "dog_id",
        "category",
        "frame_count",
        "detail",
    ],
)
print("Video status:")
display(videos_report["status"].value_counts().rename("videos"))
print("Event status:")
display(events_report["status"].value_counts().rename("events"))
print(f"Scored {len(scored)} detected events. Unscored detections and affected videos are below.")
display(videos_report[videos_report["status"] != "ok"])
display(events_report[events_report["status"] != "scored"])

## 4. Coverage by labeled span and category

An annotation with zero scored events remains in this report. A result can be absent because the source has no detection, the detected event crossed a boundary, video frames were unavailable, or an error occurred. Review the video and event status tables above before interpreting scores.


In [ ]:
if events_report.empty:
    matched_counts = pd.DataFrame(columns=["label_row", "scored_events"])
else:
    matched_counts = (
        events_report[events_report["status"] == "scored"]
        .groupby("label_row")
        .size()
        .rename("scored_events")
        .reset_index()
    )
coverage = labels.merge(matched_counts, on="label_row", how="left")
coverage["scored_events"] = coverage["scored_events"].fillna(0).astype(int)
coverage = coverage.merge(
    videos_report[["video_key", "status"]].rename(columns={"status": "video_status"}),
    on="video_key",
    how="left",
)
coverage["has_scored_event"] = coverage["scored_events"] > 0
coverage_by_category = (
    coverage.groupby("category")
    .agg(
        labeled_spans=("label_row", "size"),
        spans_scored=("has_scored_event", "sum"),
        scored_events=("scored_events", "sum"),
        dogs=("dog_id", "nunique"),
    )
    .reindex(CATEGORIES, fill_value=0)
)
display(coverage_by_category)
display(
    coverage[
        [
            "label_row",
            "video_id",
            "dog_id",
            "start_s",
            "end_s",
            "category",
            "video_status",
            "scored_events",
        ]
    ]
)

## 5. Compare aggregations on exactly the same events

Confidence is the **highest category score for each frame**, not the segment's majority class. Weight `w = (1 - alpha) + alpha * confidence`. Temperature affects both the weights and the final softmax for the weighted method. Mode is the fraction of frame votes for each category; ties use taxonomy order for the displayed winner and are flagged. It is not a calibrated probability.


In [ ]:
def softmax(values, axis=-1):
    values = np.asarray(values, dtype=np.float64)
    exp = np.exp(values - values.max(axis=axis, keepdims=True))
    return exp / exp.sum(axis=axis, keepdims=True)


def aggregate(logits, temperature, alpha):
    frame_scores = softmax(logits / temperature)
    weights = (1 - alpha) + alpha * frame_scores.max(axis=1)
    return {
        "mean_frame_probabilities": frame_scores.mean(axis=0),
        "mean_frame_logits": softmax(logits.mean(axis=0) / temperature),
        "confidence_weighted_logits": softmax(
            np.average(logits, axis=0, weights=weights) / temperature
        ),
    }


prediction_rows = []
for event in scored:
    logits = event["frame_logits"]
    # Ensure the comparison reproduces the actual configured scene branch.
    production = aggregate(logits, scene.encoder.temperature, scene.encoder.confidence_weight)
    np.testing.assert_allclose(
        production[scene.encoder.aggregation], event["configured_probs"], atol=1e-6, rtol=1e-5
    )
    record = {
        key: event[key]
        for key in [
            "video_id",
            "start_s",
            "end_s",
            "label_row",
            "dog_id",
            "category",
            "frame_count",
        ]
    }
    for temperature in TEMPERATURES:
        for method, scores in aggregate(logits, temperature, CONFIDENCE_WEIGHT).items():
            prediction_rows.append(
                {
                    **record,
                    "method": method,
                    "temperature": temperature,
                    "prediction": CATEGORIES[int(scores.argmax())],
                    "scores": scores,
                    "vote_tie": False,
                }
            )
    counts = np.bincount(logits.argmax(axis=1), minlength=len(CATEGORIES))
    votes = counts / counts.sum()
    prediction_rows.append(
        {
            **record,
            "method": "mode_votes",
            "temperature": np.nan,
            "prediction": CATEGORIES[int(votes.argmax())],
            "scores": votes,
            "vote_tie": bool(np.count_nonzero(counts == counts.max()) > 1),
        }
    )

predictions = pd.DataFrame(
    prediction_rows,
    columns=[
        "video_id",
        "start_s",
        "end_s",
        "label_row",
        "dog_id",
        "category",
        "frame_count",
        "method",
        "temperature",
        "prediction",
        "scores",
        "vote_tie",
    ],
)
if predictions.empty:
    print(
        "No scored events. Inspect the coverage and error reports above; no metrics can be computed."
    )
else:
    print(f"{len(scored)} identical scored events in every method; sample predictions:")
    display(predictions.drop(columns="scores").head(20))
    print("Example five-score vectors in taxonomy order:")
    score_view = pd.DataFrame(predictions["scores"].tolist(), columns=CATEGORIES)
    display(
        pd.concat(
            [
                predictions[
                    ["video_id", "start_s", "method", "temperature", "category", "prediction"]
                ].reset_index(drop=True),
                score_view,
            ],
            axis=1,
        ).head(20)
    )
    print(
        "Mode ties:", int(predictions.loc[predictions["method"] == "mode_votes", "vote_tie"].sum())
    )

## 6. Compare segment-level results

Accuracy and macro-F1 use the span category as a weak event label. Macro-F1 is averaged over the categories **present in the scored reference labels**; the table also shows counts so missing categories are visible. Each detected event gets equal weight, meaning videos with many detections contribute more. Compare coverage and dogs per class; these are not independent frame-level observations. Log loss and multiclass Brier score are descriptive, uncalibrated probability-quality metrics; they are omitted for hard voting.


In [ ]:
if not predictions.empty:
    metrics = []
    # dropna=False keeps the single mode row, which has no meaningful temperature.
    for (method, temperature), group in predictions.groupby(
        ["method", "temperature"], dropna=False, sort=False
    ):
        truth, guessed = group["category"].to_numpy(), group["prediction"].to_numpy()
        present_classes = [c for c in CATEGORIES if c in set(truth)]
        scores = np.stack(group["scores"].to_numpy())
        target = np.eye(len(CATEGORIES))[[CATEGORIES.index(label) for label in truth]]
        metrics.append(
            {
                "method": method,
                "temperature": temperature,
                "events": len(group),
                "dogs": group["dog_id"].nunique(),
                "accuracy": accuracy_score(truth, guessed),
                "macro_f1_present_classes": f1_score(
                    truth, guessed, labels=present_classes, average="macro", zero_division=0
                ),
                "log_loss": np.nan
                if method == "mode_votes"
                else -np.log(
                    np.clip((scores * target).sum(axis=1), np.finfo(float).tiny, 1)
                ).mean(),
                "brier_multiclass": np.nan
                if method == "mode_votes"
                else np.mean(np.sum((scores - target) ** 2, axis=1)),
            }
        )
    metrics_table = pd.DataFrame(metrics)
    display(metrics_table)
    print("Per-category recall; columns also show the number of events and dogs for each category:")
    recall_rows = []
    for (method, temperature), group in predictions.groupby(
        ["method", "temperature"], dropna=False, sort=False
    ):
        for category in CATEGORIES:
            subset = group[group["category"] == category]
            recall_rows.append(
                {
                    "method": method,
                    "temperature": temperature,
                    "category": category,
                    "events": len(subset),
                    "dogs": subset["dog_id"].nunique(),
                    "recall": (subset["prediction"] == category).mean() if len(subset) else np.nan,
                }
            )
    display(pd.DataFrame(recall_rows))
    print(
        "Inspect confusion matrices for every method (rows = labeled span, columns = prediction):"
    )
    for (method, temperature), group in predictions.groupby(
        ["method", "temperature"], dropna=False, sort=False
    ):
        print(f"{method}, T={temperature}")
        matrix = confusion_matrix(group["category"], group["prediction"], labels=CATEGORIES)
        display(pd.DataFrame(matrix, index=CATEGORIES, columns=CATEGORIES))
else:
    print("No predictions to evaluate.")

Results are descriptive for these testing videos. For method selection, use a separate dog-disjoint validation collection, select settings once, then evaluate once on held-out dogs. Inspect videos for disagreements, especially boundary-crossing events, repeated frames, wrong detections and incorrect or ambiguous span labels. The app's fusion pipeline is separate and does not run in this notebook.
